# HuggingFace TGI + AuditKIT: evaluate a model served by TGI

TGI serves a model behind an OpenAI-compatible HTTP API, and AuditKIT's `api:` backend evaluates it. AuditKIT never imports TGI: it POSTs to a `/v1` base URL and reads the reply. Its own environment needs only `auditkit[requests]`, which is why the two halves can live in completely separate environments — the container below and this kernel.

> **This notebook needs a running TGI server.** It ships with empty outputs on purpose: no TGI endpoint was available where it was written, so nothing here has been executed. Every cell is real code and will run once a server is up.
>
> Start one first (Linux + NVIDIA GPU):
>
> ```bash
> docker run --gpus all --shm-size 1g -p 8080:80 \
>   -v "$HOME/.cache/huggingface:/data" \
>   ghcr.io/huggingface/text-generation-inference:3.3.7 \
>   --model-id CohereLabs/tiny-aya-global --json-output
> ```
>
> Colab cannot run TGI's GPU stack in-process, but it can expose the server's HTTP port: the launch cell calls `output.serve_kernel_port_as_window(PORT)`. On a GPU host, delete that branch and just start the container.

| § | What |
|---|---|
| 1 | What you need, and the install |
| 2 | Start (or point at) a TGI server and wait until it is healthy |
| 3 | Point AuditKIT at it, and print the URL it will actually POST to |
| 4 | One generation eval |
| 5 | One judged metric, with the same server as the judge |
| 6 | Tool calling in prompt mode (the Cohere templates have no tool slot) |
| 7 | What to check when it fails |

**Access:** Tiny Aya is gated, so the server needs `HF_TOKEN` for the download. To use an open model instead, set `MODEL = "Qwen/Qwen2.5-1.5B-Instruct"` in the settings cell — that one also has a native tool parser worth comparing against.

Full writeup: [`docs/HF_ENDPOINT.md`](https://github.com/Lexsi-Labs/AuditKIT/blob/main/docs/HF_ENDPOINT.md).

## 1 · What you need

- A running TGI server (see the banner above), reachable at a `/v1` base URL.
- `auditkit[requests]` here. Nothing else — no `torch`, no `transformers`, no TGI.
- An `api_key` only if the server was started with `--api-key`. TGI ignores the bearer token otherwise, so any string works.

In [ ]:
# ---- install (Colab) / reuse (local checkout) ----
# Colab: clones AuditKIT at REPO_BRANCH and installs it with the extras this notebook needs.
# Local: uses the auditkit already importable (e.g. the repo's .venv kernel).
import os, sys, subprocess

REPO_BRANCH = "feat/multipath-tool-references"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import userdata
    for key in ("GITHUB_TOKEN", "HF_TOKEN"):
        try:
            os.environ[key] = userdata.get(key)
        except Exception:
            pass
    repo_url = f"https://{os.environ['GITHUB_TOKEN']}@github.com/aryapratinavseth/AuditKIT-Internal.git"
    if not os.path.isdir("/content/AuditKIT-Internal"):
        subprocess.run(["git", "clone", "-q", "--branch", REPO_BRANCH, repo_url, "/content/AuditKIT-Internal"], check=True)
    else:   # a reused runtime: move the existing clone to the branch head, never run stale code
        subprocess.run(["git", "-C", "/content/AuditKIT-Internal", "fetch", "-q", repo_url, REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", "/content/AuditKIT-Internal", "checkout", "-q", "-B", REPO_BRANCH, "FETCH_HEAD"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "-e",
                    "/content/AuditKIT-Internal[requests]"], check=True)
    sys.path.insert(0, "/content/AuditKIT-Internal/src")

# ---- server coordinates ----
# MODEL is what --model-id was given on the server. It is only used for run labels here:
# TGI serves one model per process and AuditKIT posts model="tgi" (see section 3).
MODEL = os.environ.get("NB_MODEL", "CohereLabs/tiny-aya-global")
PORT = 8080
API_BASE = f"http://127.0.0.1:{PORT}/v1"     # the /v1 base -- api: appends the rest itself
API_KEY = os.environ.get("TGI_API_KEY") or "EMPTY"   # any string unless the server used --api-key

import requests
import auditkit as ak
from auditkit import RunConfig, Sample
from auditkit.model.api_gen import APIModel
from auditkit.model import AutoModel

print("auditkit", ak.__version__, "| serving:", MODEL, "| api_base:", API_BASE)

## 2 · Start or point at a server

Skip this cell if you already have TGI running — go straight to section 3. Otherwise this starts the container and blocks until `/health` answers 200. Colab has no nested Docker, so on Colab you forward a port to a server you started elsewhere and delete the `launch()` body.

In [ ]:
SERVER = None

def launch():
    """Start the TGI container in the background (Linux + NVIDIA GPU host only)."""
    image = "ghcr.io/huggingface/text-generation-inference:3.3.7"
    cmd = ["docker", "run", "--rm", "--gpus", "all", "--shm-size", "1g",
           "-p", f"{PORT}:80", "-v", f"{os.path.expanduser('~')}/.cache/huggingface:/data", image,
           "--model-id", MODEL, "--json-output"]
    if os.environ.get("TGI_API_KEY"):
        cmd += ["--api-key", os.environ["TGI_API_KEY"]]
    print("launching:", " ".join(cmd[6:]))
    return subprocess.Popen(cmd, stdout=open("/tmp/tgi.log", "w"), stderr=subprocess.STDOUT,
                            start_new_session=True, env={**os.environ, **({"HF_TOKEN": os.environ["HF_TOKEN"]} if os.environ.get("HF_TOKEN") else {})})

def log_tail(n=40):
    try:
        return "".join(open("/tmp/tgi.log").readlines()[-n:])
    except OSError:
        return "(no log -- the server was not started from this kernel)"

def wait_healthy(proc, timeout_s=900):
    """True once /health answers 200; False if the process exited; raises on timeout."""
    import time
    deadline = time.time() + timeout_s
    while True:
        if proc is not None and proc.poll() is not None:
            return False
        try:
            if requests.get(f"http://127.0.0.1:{PORT}/health", timeout=5).status_code == 200:
                return True
        except requests.RequestException:
            pass
        if time.time() > deadline:
            raise TimeoutError("server not healthy in time:\n" + log_tail())
        time.sleep(5)

if not IN_COLAB and os.environ.get("NB_SKIP_LAUNCH") != "1":
    SERVER = launch()
elif IN_COLAB:
    from google.colab import output
    output.serve_kernel_port_as_window(PORT)   # optional: clickable endpoint
    print("Colab: forward the port from the machine running TGI, e.g.")
    print("  ssh -N -L 8080:localhost:8080 user@gpu-host")
else:
    print("NB_SKIP_LAUNCH=1: assuming a TGI server is already running")

if wait_healthy(SERVER):
    info = requests.get(f"{API_BASE}/models", timeout=10)
    print("server healthy:", info.json() if info.ok else info.status_code)
    print("/info:", requests.get(f"http://127.0.0.1:{PORT}/info", timeout=10).text[:300])

## 3 · Point AuditKIT at it

There is no `tgi:` spec prefix — `api:` is the whole integration. Two things to get right:

- **`api_base` must end in `/v1`.** `APIModel` strips a trailing slash and appends `/chat/completions` itself; it never adds `/v1` for you.
- **`model="tgi"`.** TGI serves one model per process and its own examples send `{"model": "tgi"}`. This is the opposite of vLLM/SGLang, where you name the served checkpoint — see [HF_ENDPOINT.md](https://github.com/Lexsi-Labs/AuditKIT/blob/main/docs/HF_ENDPOINT.md).

`api_base` and `chat_template` are part of the run fingerprint, so pointing the same `api:` spec at a different server never replays a stale cached result.

In [ ]:
tgi = APIModel(model="tgi", api_base=API_BASE, api_key=API_KEY, name="tgi")
print("resolved base :", tgi._api_base)
print("will POST to  :", tgi._api_base + "/chat/completions")
print("capabilities  :", sorted(c.value for c in tgi.capabilities()))
print("identity      :", tgi.identity())

# One POST, by hand, so a failure here is the server's and not AuditKIT's.
probe = requests.post(f"{API_BASE}/chat/completions", timeout=60, headers={
    "Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"},
    json={"model": "tgi", "messages": [{"role": "user", "content": "Reply with the single word: ok"}],
          "max_tokens": 16, "temperature": 0.0})
print("probe", probe.status_code, "->", probe.text[:300])

# The spec-string form is identical -- evaluate() forwards api_base=/api_key= to the same constructor.
spec_form = AutoModel.resolve("api:tgi", api_base=API_BASE, api_key=API_KEY)
print("spec form     :", spec_form.name, "|", spec_form._model_name, "|", spec_form._api_base)

## 4 · One generation eval

Reuse the `tgi` object above so the model is built once. `RunConfig.temperature` defaults to 0.0, which TGI reads as *greedy* (sampling off), so an eval is deterministic by default.

In [ ]:
QA = [("What is the capital of France? Answer with one word.", "Paris"),
      ("Translate to Spanish, reply with the translation only: thank you", "gracias"),
      ("Wie viele Tage hat eine Woche? Nur die Zahl.", "7"),
      ("日本の首都はどこですか？一語で答えてください。", "東京"),
      ("What is 12 times 12? Answer with the number only.", "144"),
      ("¿Cuál es la capital de Perú? Responde con una palabra.", "Lima")]
qa = [Sample(input=q, target=a) for q, a in QA]

gen = ak.evaluate(qa, model=tgi,
                  scorers=["quasi_exact_match", "f1_score", "chrf"],
                  config=RunConfig(temperature=0.0, max_tokens=48, concurrency=4))
print("headline:", {k: round(v, 3) for k, v in gen.headline.items()}, "| errors:", gen.errors)
for p in gen.predictions:
    print(f"  {p.expected!r:>12} <- {(p.raw_output or '').strip()[:60]!r}")
print(gen.summary())       # latency / throughput measured over HTTP

# APIModel surfaces the server's response body, not a bare "400 Client Error", so
# gen.errors usually says exactly what TGI objected to (input too long, bad field, ...).
for err in gen.errors:
    print("run error:", str(err)[:400])

## 5 · One judged metric

A judge is just another model spec. `judge_model_args` carries the connection details (`api_base`/`api_key` here), so the same TGI server can judge the model's own answers. Generation settings for the judge go on the metric (`temperature=`, `max_tokens=`), never on the connection args.

In [ ]:
open_qa = [
    Sample(input="Explain why the sky looks blue to a ten-year-old.", target=""),
    Sample(input="I have 3 apples and eat 2, then buy 5 more. How many apples do I have?", target=""),
    Sample(input="What is 2+2? Reply with only the number.", target=""),
]

judged = ak.evaluate(
    open_qa, model=tgi,
    scorers=[ak.LLMJudge(judge_model="api:tgi",
                       judge_model_args={"api_base": API_BASE, "api_key": API_KEY},
                       name="clarity", temperature=0.0, max_tokens=16)],
    config=RunConfig(temperature=0.0, max_tokens=128),
)
print("headline:", {k: round(v, 3) for k, v in judged.headline.items()}, "| errors:", judged.errors)
for p in judged.predictions:
    print(f"  {p.sample_id}: score={p.score} {(p.raw_output or '').strip()[:80]!r}")

## 6 · Tool calling in prompt mode

TGI's chat route does accept `tools` and `tool_choice`, so `ToolCallAdapter()` (native mode) has a server-side counterpart — but that is not what the four Cohere hackathon models (Tiny Aya, Aya Expanse, Aya Vision, North) can use: **none of their chat templates renders a `tools` slot**, so there is nothing for any server to render the schemas into.

`ToolCallAdapter(mode="prompt")` writes the schemas into the system message and asks for `<tool_call>` blocks, then parses them out of the reply. Swap `MODEL` for a model with a tool-rendering template (Qwen2.5 works) and use the plain `ToolCallAdapter()` instead — the calls then come back structured and the metrics read them from `context["trace"]`.

In [ ]:
TOOLS = [{"type": "function", "function": {"name": "get_weather", "description": "Current weather for a city",
          "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}},
         {"type": "function", "function": {"name": "get_time", "description": "Current local time in a city",
          "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}}]
W = lambda c: {"name": "get_weather", "arguments": {"city": c}}
tool_tasks = [Sample(id="single", input="What's the weather in Paris?", tools=TOOLS, expected_tool_calls=[W("Paris")]),
              Sample(id="parallel", input="What's the weather in Paris and in Rome? Call the tool for both cities.",
                     tools=TOOLS, expected_tool_calls=[[W("Paris"), W("Rome")]]),
              Sample(id="no_tool", input="What is 2+2? Answer directly, no tools.", tools=TOOLS, expected_tool_calls=[]),
              Sample(id="pick", input="What time is it in Tokyo?", tools=TOOLS,
                     expected_tool_calls=[{"name": "get_time", "arguments": {"city": "Tokyo"}}])]

tools_run = ak.evaluate(tool_tasks, model=tgi,
                        adapter=ak.ToolCallAdapter(mode="prompt"),
                        scorers=[ak.ToolCallF1(), ak.ParallelToolCalls(), ak.ToolCallValidity()],
                        config=RunConfig(temperature=0.0, max_tokens=256))
print("headline:", {k: round(v, 3) for k, v in tools_run.headline.items()}, "| errors:", tools_run.errors)
for p in tools_run.predictions:
    print(f"  {p.sample_id:9s} {(p.raw_output or '').strip()[:110]!r}")

# What the metrics parsed back out of the text:
print(ak.parse_tool_calls(tools_run.predictions[0].raw_output or ""))

## 7 · What to check when it fails

Most `api:`-at-TGI failures are one of these five, and each has a different fix.

In [ ]:
def diagnose():
    base = f"http://127.0.0.1:{PORT}"
    # 1. is anything listening / did the download finish?
    for path in ("/health", "/info", "/v1/models"):
        try:
            r = requests.get(base + path, timeout=10)
            print(f"{path:12} {r.status_code} {r.text[:160]}")
        except requests.RequestException as e:
            print(f"{path:12} unreachable ({type(e).__name__}) -- server not started yet, or the port is not forwarded")
    # 2. auth: only enforced when the server was launched with --api-key
    r = requests.post(f"{API_BASE}/chat/completions", timeout=30, headers={
        "Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"},
        json={"model": "tgi", "messages": [{"role": "user", "content": "ping"}], "max_tokens": 8})
    print("auth probe  ", r.status_code, r.text[:200])
    # 3. the base URL: a missing /v1 is a 404 on a perfectly healthy server
    r2 = requests.post(f"http://127.0.0.1:{PORT}/chat/completions", timeout=30, headers={
        "Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"},
        json={"model": "tgi", "messages": [{"role": "user", "content": "ping"}], "max_tokens": 8})
    print("no /v1 base", r2.status_code, "-- if this is 404 and the /v1 probe worked, api_base is missing /v1")
    # 4. server log, for OOM / unsupported-arch / gated-download failures
    print("\nserver log tail:\n" + log_tail(30))

diagnose()

**Reading the results.**

| Symptom | Likely cause | Fix |
|---|---|---|
| `ConnectionError` on the first request | nothing listening, or a Colab port not forwarded | section 2 / 7 |
| HTTP 404 on every request | `api_base` is missing the `/v1` segment | `API_BASE = f"http://127.0.0.1:{PORT}/v1"` |
| HTTP 401 | server started with `--api-key`, wrong token passed | set `TGI_API_KEY` before launching and in `API_KEY` |
| HTTP 422, mentions input/parameters | prompt + `max_tokens` exceeds `--max-input-tokens` / `--max-total-tokens`, or `n`/`stop` over the server's limits | shrink `RunConfig.max_tokens`, or raise the flags at launch |
| `no choices in response` / `no message in first choice` | the reply is not OpenAI-shaped (proxied error page, truncated body) | print `resp.text` in the probe above |
| Model answers, but ignores the system prompt or turn markers | `chat_template=False` posted raw text, or the model has no usable chat template | keep the default `chat_template=True` |
| Every tool metric is 0 and the model answers in prose | the template has no `tools` slot (all four Cohere models) | `ToolCallAdapter(mode="prompt")`, section 6 |
| A template switch (`enable_thinking`) seems to do nothing | TGI's chat request ignores `chat_template_kwargs` | verify the model's output actually changed; use a model that needs no switch |

**Shut down** (only if this kernel started the server):

```python
if SERVER is not None:
    import signal
    os.killpg(os.getpgid(SERVER.pid), signal.SIGTERM)
    SERVER.wait(timeout=60)
    print("server stopped")
```